In [1]:
from langgraph.graph import StateGraph , START, END
from langchain_core.messages import SystemMessage, HumanMessage , BaseMessage
from langchain_huggingface import HuggingFacePipeline, HuggingFaceEndpoint, ChatHuggingFace
from dotenv import load_dotenv
from typing import TypedDict , Annotated
import operator 
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver



In [2]:
load_dotenv()

True

In [30]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    temperature=0.7,
)

model = ChatHuggingFace(llm=llm)

In [31]:
class WorkflowState(TypedDict):
    messages:Annotated[list[BaseMessage],add_messages ]
    

In [33]:
def chat_message(state:WorkflowState) -> WorkflowState:
       messages = state['messages']
       response=model.invoke(messages)
       return {'messages': [response]}
       

In [34]:

checkpointer=MemorySaver();
stategraph=StateGraph(WorkflowState)

stategraph.add_node("chatmeessage",chat_message)

stategraph.add_edge(START,"chatmeessage"),
stategraph.add_edge("chatmeessage",END)
chatbot=stategraph.compile(checkpointer=checkpointer)


In [35]:
thread_id='1'
while True:
    user_input = input("User: ")
    if user_input.lower() == "exit":
        break
    print(f"User: {user_input}")
    initial_state = {
        'messages': [HumanMessage(content=user_input)]
    }
    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }
    response = chatbot.invoke(initial_state, config=config)
    print(f"Chatbot: {response['messages'][-1].content}")

KeyboardInterrupt: Interrupted by user

In [17]:
initial_state = {
    'messages': [HumanMessage(content='What is the capital of india')]
}

chatbot.invoke(initial_state)['messages'][-1].content

'The capital of India is New Delhi. It is located in the northern part of the country and is known for its rich history, cultural landmarks, and governmental institutions.'